# 店舗売上需要予測: 実験 #003 ラグ（過去実績）& 移動平均モデル

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** テスト期間（16日間）を考慮したリーク完全ゼロの「過去売上実績（Lag 16〜）」と「ローリング移動平均」による大躍進
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **モデル:** **LightGBM Regressor**
- **現在自己ベスト:** 実験 #001: **LB 0.52338**

---

## 1. 時系列予測の最重要ルール：未来リークを防ぐラグ設計

### なぜ「1日前の売上（Lag 1）」を使ってはいけないのか？
- テスト期間は **2017-08-16 〜 2017-08-31（16日間）** です。
- テストの最終日（8月31日）の予測を行うとき、手元にある最新の確定売上は「8月15日（16日前）」です。
- もし安易に `Lag 1` や `Lag 7` を作ってしまうと、テスト期間の2日目以降（8月17日〜31日）では未来のデータが存在しないため `NaN` になるか、テストデータをカンニングする致命的リークになります。

### 今回採用する安全で強力なラグ & ローリング特徴量
店舗（`store_nbr`）× 商品ファミリー（`family`）の系列ごとに計算：
1. **`sales_lag_16`:** 16日前の売上（テスト全期間で必ず存在する直近実績）
2. **`sales_lag_21`:** 21日前の売上（**ちょうど3週間前の同じ曜日！**）
3. **`sales_lag_28`:** 28日前の売上（**ちょうど4週間前の同じ曜日！**）
4. **`sales_lag_35`:** 35日前の売上（**ちょうど5週間前の同じ曜日！**）
5. **`sales_rolling_mean_7` / `14` / `28`:** 16日前の売上を起点とした直近7日・14日・28日の移動平均需要水準
6. **`sales_rolling_std_7`:** 直近7日間の売上のばらつき（需要の安定度）

※実験 #002 でノイズとなった原油価格や店舗冗長列（都市・州など）はアブレーション（削ぎ落とし）しています。

## 2. ライブラリ読み込み & データ読み込み

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# データディレクトリ自動検出
def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        '../data',
        'data'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    if os.path.exists('/kaggle/input'):
        input_contents = os.listdir('/kaggle/input')
        raise FileNotFoundError(f'/kaggle/input にデータがありません。現在の中身: {input_contents}。右上の [+ Add Input] から "store-sales-time-series-forecasting" を追加してください。')
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sub_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

# ラグ特徴量作成のため train と test を縦結合（testの sales は NaN）
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])

print(f'全体データ形状: {all_df.shape} (期間: {all_df["date"].min().strftime("%Y-%m-%d")} 〜 {all_df["date"].max().strftime("%Y-%m-%d")})')

## 3. カレンダー ＆ 給与日特徴量

In [ ]:
all_df['year'] = all_df['date'].dt.year
all_df['month'] = all_df['date'].dt.month
all_df['day'] = all_df['date'].dt.day
all_df['dayofweek'] = all_df['date'].dt.dayofweek
all_df['is_weekend'] = all_df['dayofweek'].isin([5, 6]).astype(int)
all_df['dayofyear'] = all_df['date'].dt.dayofyear
all_df['weekofyear'] = all_df['date'].dt.isocalendar().week.astype(int)
all_df['is_month_start'] = all_df['date'].dt.is_month_start.astype(int)
all_df['is_month_end'] = all_df['date'].dt.is_month_end.astype(int)
all_df['is_wage_day'] = ((all_df['day'] == 15) | (all_df['is_month_end'] == 1)).astype(int)
all_df['is_wage_day_after'] = ((all_df['day'] == 16) | (all_df['is_month_start'] == 1)).astype(int)
print('カレンダー & 給与日特徴量作成完了')

## 4. 時系列ラグ（過去売上）＆ ローリング移動平均の構築

In [ ]:
# 店舗 × 商品ファミリー × 日付順にソート
all_df = all_df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
group = all_df.groupby(['store_nbr', 'family'])['log_sales']

print('ラグ特徴量作成中 (Lag 16, 21, 28, 35)...')
for lag in [16, 21, 28, 35]:
    all_df[f'sales_lag_{lag}'] = group.shift(lag)

print('ローリング移動平均作成中 (16日前起点の 7日, 14日, 28日平均)...')
lag16_series = group.shift(16)
all_df['sales_rolling_mean_7'] = lag16_series.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
all_df['sales_rolling_mean_14'] = lag16_series.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(14, min_periods=1).mean()
)
all_df['sales_rolling_mean_28'] = lag16_series.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(28, min_periods=1).mean()
)
all_df['sales_rolling_std_7'] = lag16_series.groupby([all_df['store_nbr'], all_df['family']]).transform(
    lambda x: x.rolling(7, min_periods=1).std()
).fillna(0)

# カテゴリ変数の型変換
for col in ['store_nbr', 'family']:
    all_df[col] = all_df[col].astype('category')

# 訓練データとテストデータに再分離
train_feat = all_df[all_df['date'] <= '2017-08-15'].copy()
test_feat = all_df[all_df['date'] >= '2017-08-16'].copy()

ignore_cols = ['id', 'date', 'sales', 'log_sales']
features = [c for c in all_df.columns if c not in ignore_cols]

print(f'総特徴量数: {len(features)} 列')
print(f'特徴量一覧: {features}')

## 5. 時系列バリデーション分割 (Time-based Split)

In [ ]:
val_start = '2017-07-31'
val_end = '2017-08-15'
train_start = '2016-01-01'

trn_mask = (train_feat['date'] >= train_start) & (train_feat['date'] < val_start)
val_mask = (train_feat['date'] >= val_start) & (train_feat['date'] <= val_end)

X_train, y_train = train_feat.loc[trn_mask, features], train_feat.loc[trn_mask, 'sales'].values
X_val, y_val = train_feat.loc[val_mask, features], train_feat.loc[val_mask, 'sales'].values
X_test = test_feat[features]

print(f'学習サンプル数: {len(X_train):,} 件')
print(f'検証サンプル数: {len(X_val):,} 件')

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

## 6. LightGBM 回帰モデル学習

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'random_state': 42,
    'verbose': -1,
    'n_estimators': 1000,
}

model = lgb.LGBMRegressor(**params)
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100)
]

model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=callbacks,
)

# 特徴量重要度 Top 15 の可視化
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#2ca02c')
plt.title('LightGBM Feature Importance (Top Features)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. 手元検証スコア（RMSLE）の評価

In [ ]:
val_pred_log = model.predict(X_val)
val_pred = np.clip(np.expm1(val_pred_log), 0, None)

rmsle = np.sqrt(mean_squared_error(np.log1p(y_val), np.log1p(val_pred)))
print('=' * 60)
print(f'  🏆 [実験 #003] 手元検証スコア (Val RMSLE): {rmsle:.5f}')
print(f'  (参考: #001 ベースライン: 0.41440 / #002 ドメイン: 0.41426)')
print('=' * 60)

# 実測値 vs 予測値の推移プロット
val_df = train_feat.loc[val_mask].copy()
val_df['pred_sales'] = val_pred
daily_val = val_df.groupby('date')[['sales', 'pred_sales']].sum().reset_index()

plt.figure(figsize=(12, 4))
plt.plot(daily_val['date'], daily_val['sales'], label='Actual (y_true)', marker='o', color='#1f77b4')
plt.plot(daily_val['date'], daily_val['pred_sales'], label='Predicted (y_pred)', marker='x', linestyle='--', color='#ff7f0e')
plt.title(f'Validation Actual vs Predicted (RMSLE: {rmsle:.5f})')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. テストデータ予測 & 提出ファイル作成

In [ ]:
test_pred_log = model.predict(X_test)
test_pred = np.clip(np.expm1(test_pred_log), 0, None)

# 重要: test_feat はラグ計算時にソートされているため、id順に正しく揃えて提出する
test_feat['sales'] = test_pred
submission_df = test_feat[['id', 'sales']].sort_values('id').reset_index(drop=True)
submission_df.to_csv('submission.csv', index=False)

print('提出用 submission.csv を作成しました（id順ソート完了）！')
print(f'行数: {len(submission_df):,} 行')
print(submission_df.head(10))
print(submission_df.describe())